# Laboratorio 6 — Hito 1: carga, integración, calidad y preprocesamiento

**Universidad del Valle de Guatemala — CC3084 Data Science — Semestre II, 2026**

Cubre las actividades 1 y 2 del enunciado.

## tl;dr

- Se cargaron 293 videos y 406 comentarios sin modificar los archivos originales.
- Las llaves primarias son completas y únicas; los 406 comentarios se asocian con un video sin pérdida ni expansión de filas.
- El riesgo principal no es la integridad de la unión sino la cobertura: sólo 19 de 293 videos (6.48 %) tienen comentarios recolectados.
- El diagnóstico detecta 189 conteos de «me gusta» en blanco, 27 handles con codificación porcentual de URL y dos variables inutilizables (viewer_rating, is_pinned).
- La limpieza de texto no elimina ningún registro y reduce el volumen de tokens en 61.2 %.

## 0. Preparación del entorno

In [1]:
from __future__ import annotations

import json
import re
import unicodedata
import warnings
from urllib.parse import unquote
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from scipy import stats

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titleweight": "bold",
})

AZUL, NARANJA, VERDE, ROJO, MORADO, GRIS = (
    "#2b6cb0", "#dd6b20", "#2f855a", "#c53030", "#6b46c1", "#718096",
)
PALETA = [AZUL, NARANJA, VERDE, ROJO, MORADO, "#00838f", "#b7791f", "#4a5568"]


def encontrar_raiz(inicio: Path) -> Path:
    """Localiza la raíz del repositorio buscando los CSV originales."""
    for candidato in (inicio, *inicio.parents):
        if (candidato / "youtube_videos.csv").exists() and (candidato / "youtube_comments.csv").exists():
            return candidato
    raise FileNotFoundError("No se encontraron youtube_videos.csv y youtube_comments.csv.")


try:
    BASE = Path(__file__).resolve().parent
except NameError:  # ejecución interactiva / notebook
    BASE = Path.cwd()

ROOT = encontrar_raiz(BASE)
TABLAS = ROOT / "outputs" / "tables"
FIGURAS = ROOT / "outputs" / "figures"
GRAFOS = ROOT / "outputs" / "graphs"
PROCESADOS = ROOT / "data" / "processed"
for carpeta in (TABLAS, FIGURAS, GRAFOS, PROCESADOS):
    carpeta.mkdir(parents=True, exist_ok=True)

RESULTADOS: dict = {}


def registrar(clave: str, valor) -> None:
    """Guarda una métrica para reutilizarla en el informe escrito."""
    RESULTADOS[clave] = valor


def guardar_tabla(frame: pd.DataFrame, nombre: str) -> pd.DataFrame:
    frame.to_csv(TABLAS / f"{nombre}.csv", index=False, encoding="utf-8-sig")
    return frame


def guardar_figura(nombre: str) -> None:
    plt.savefig(FIGURAS / f"{nombre}.png")
    plt.show()          # muestra la figura en el notebook; en modo script no hace nada
    plt.close()


def acortar(valor, ancho: int = 46) -> str:
    valor = str(valor)
    return valor if len(valor) <= ancho else valor[: ancho - 1] + "…"


print(f"Raíz del proyecto: {ROOT}")

Raíz del proyecto: /Users/javiervalladares/Library/Mobile Documents/com~apple~CloudDocs/Octavo Semestre/Datos/Lab6/repo


## 1. Carga, comprensión e integración de los datos

### 1.1 Carga de `youtube_videos.csv` y `youtube_comments.csv`

Los archivos se leen con `encoding="utf-8-sig"` porque traen BOM. Todas las columnas que son
identificadores se fuerzan a texto: si pandas las infiriera como numéricas podría perder ceros a
la izquierda o convertir un ID a notación científica.

In [2]:
COLS_ID_VIDEOS = ["video_id", "channel_id", "channel_handle", "owner_handle"]
COLS_ID_COMENTARIOS = ["video_id", "comment_id", "channel_id", "author_channel_id", "author_handle"]

videos_raw = pd.read_csv(
    ROOT / "youtube_videos.csv",
    encoding="utf-8-sig",
    dtype={c: "string" for c in COLS_ID_VIDEOS},
    keep_default_na=True,
)
comentarios_raw = pd.read_csv(
    ROOT / "youtube_comments.csv",
    encoding="utf-8-sig",
    dtype={c: "string" for c in COLS_ID_COMENTARIOS},
    keep_default_na=True,
)

registrar("n_videos", int(len(videos_raw)))
registrar("n_comentarios", int(len(comentarios_raw)))
registrar("n_vars_videos", int(videos_raw.shape[1]))
registrar("n_vars_comentarios", int(comentarios_raw.shape[1]))

print(f"youtube_videos.csv    : {videos_raw.shape[0]} filas × {videos_raw.shape[1]} variables")
print(f"youtube_comments.csv  : {comentarios_raw.shape[0]} filas × {comentarios_raw.shape[1]} variables")
videos_raw.head(3)

youtube_videos.csv    : 293 filas × 20 variables
youtube_comments.csv  : 406 filas × 17 variables


,video_id,title,channel_name,channel_id,source_query,source_group,dataset_sources,channel_handle,published_time,view_count_text,description_snippet,video_url,query_hits,keywords,description,view_count,publish_date,upload_date,category,owner_handle
0,-5puKGEqcUc,INSIVUMEH pronostica incremento de lluvias para el fin de semana en Guatemala,T13 Noticias Guatemala,UCq0Cm-3SKthEySQc2JZBi1A,guatemala lluvias,topic,youtube_guatemala.csv | youtube_guatemala_lab.csv | youtube_guatemala_plus.csv,/@T13NoticiasGuatemala,hace 2 días,"2,390 vistas",El Departamento de Pronóstico de INSIVUMEH prevé un incremento en las lluvias durante ...,https://www.youtube.com/watch?v=-5puKGEqcUc,"[""guatemala lluvias""]","[""Canícula prolongada"", ""Chapin tv"", ""Fenómeno del Niño"", ""Guatemala"", ""Lluvias en Gua...",El Departamento de Pronóstico de INSIVUMEH prevé un incremento en las lluvias durante ...,2357,2026-08-28T22:00:20-07:00,2026-08-28T22:00:20-07:00,News & Politics,/@T13NoticiasGuatemala
1,-E7OPOLjMug,BERNARDO ARÉVALO CALIFICA CAMBIO EN EL MP COMO EL FIN DE UNA ETAPA DE DETERIORO INSTIT...,IDocumenta,UCgItjn_ZWFWcv1MlpKIkqgQ,@GobiernodelaRepublicadeGuatema,topic,youtube_target_channels.csv,/@iDocumenta,hace 3 meses,4 vistas,"El presidente de Guatemala, Bernardo Arévalo, calificó el cambio de fiscal general y j...",https://www.youtube.com/watch?v=-E7OPOLjMug,"[""@GobiernodelaRepublicadeGuatema""]",[],"El presidente de Guatemala, Bernardo Arévalo, calificó el cambio de fiscal general y j...",4,2026-05-13T16:00:03-07:00,2026-05-13T16:00:03-07:00,People & Blogs,/@iDocumenta
2,-KDglrIzRKo,¡HISTÓRICO! Mexico recupera petróleo robado por Guatemala... 🔔,México Poder,UC-DpoeBbCMOMOH1-j71KPqA,guatemala noticias,topic,youtube_guatemala.csv | youtube_guatemala_lab.csv | youtube_guatemala_plus.csv,/@M%C3%A9xicoPoder,hace 1 día,"29,736 vistas",México #PetróleoMexicano #SoberaníaEnergética #Pemex #NoticiasMéxico #FronteraSur #Hua...,https://www.youtube.com/watch?v=-KDglrIzRKo,"[""guatemala noticias""]","[""Mexico recupera petroleo"", ""petroleo robado Guatemala"", ""huachicol frontera sur"", ""P...",#México #PetróleoMexicano #SoberaníaEnergética #Pemex #NoticiasMéxico #FronteraSur #Hu...,29736,2026-08-29T17:00:38-07:00,2026-08-29T17:00:38-07:00,People & Blogs,/@M%C3%A9xicoPoder


In [3]:
comentarios_raw.head(3)

,video_id,comment_id,video_title,channel_name,channel_id,author_name,author_channel_id,text,source_query,source_group,dataset_sources,author_handle,published_text,like_count_text,reply_count,is_pinned,viewer_rating
0,j43HgwYFKfk,Ugw-J65a1iYL9hqhELh4AaABAg,La cooptación de Walter Mazariegos en la USAC,Quorum,UCE4rsXcgDb6e1-a9iTbWzfg,@MarcosCarillo-b1r,UCdFlugHJJa4l3YqWuNRmvXw,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,@quorumgt,topic,j43HgwYFKfk_out_comments.csv | quorum_complete_comments.csv | youtube_guatemala_plus_c...,/@MarcosCarillo-b1r,hace 6 meses,,0,False,NaN
1,06mFNPU0aB8,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,Capturan a presuntos delincuentes disfrazados de mujer señalados de cometer asalto,Noti7,UCVpSRoZgngfSL03Nlbjtq9A,@RaulPerez-cw2vi,UCvl1tzQeBeGy6efPTRJXSCw,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay policías que le...",guatemala noticias,topic,youtube_guatemala_comments.csv | youtube_guatemala_plus_comments.csv,/@RaulPerez-cw2vi,hace 2 semanas,,0,False,NaN
2,j43HgwYFKfk,Ugw0xaOb2CYXXoudtwJ4AaABAg,La cooptación de Walter Mazariegos en la USAC,Quorum,UCE4rsXcgDb6e1-a9iTbWzfg,@iamjimalesssa,UCRAquv8el-tQ30bN7MlmySQ,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, esto no es ...",@quorumgt,topic,j43HgwYFKfk_out_comments.csv | quorum_complete_comments.csv | youtube_guatemala_plus_c...,/@iamjimalesssa,hace 1 año,4,0,False,NaN


### 1.2 Unidad de observación, llave primaria y variables relevantes

Se verifica empíricamente que las llaves candidatas son únicas y completas antes de afirmarlo.

In [4]:
def perfil_llave(frame: pd.DataFrame, columna: str) -> dict:
    serie = frame[columna]
    return {
        "archivo": None,
        "llave_candidata": columna,
        "filas": int(len(frame)),
        "valores_unicos": int(serie.nunique(dropna=True)),
        "faltantes": int(serie.isna().sum()),
        "es_llave_primaria": bool(serie.notna().all() and serie.nunique(dropna=True) == len(frame)),
    }


llaves = []
for nombre, frame, columnas in [
    ("youtube_videos.csv", videos_raw, ["video_id", "channel_id", "video_url"]),
    ("youtube_comments.csv", comentarios_raw, ["comment_id", "video_id", "author_channel_id"]),
]:
    for columna in columnas:
        fila = perfil_llave(frame, columna)
        fila["archivo"] = nombre
        llaves.append(fila)

llaves_df = guardar_tabla(pd.DataFrame(llaves), "01_llaves_candidatas")
registrar("llaves_candidatas", llaves_df.to_dict("records"))
llaves_df

,archivo,llave_candidata,filas,valores_unicos,faltantes,es_llave_primaria
0,youtube_videos.csv,video_id,293,293,0,True
1,youtube_videos.csv,channel_id,293,97,0,False
2,youtube_videos.csv,video_url,293,293,0,True
3,youtube_comments.csv,comment_id,406,406,0,True
4,youtube_comments.csv,video_id,406,19,0,False
5,youtube_comments.csv,author_channel_id,406,332,0,False


**Unidad de observación.**

| Archivo | Unidad de observación | Llave primaria | Llave foránea |
|---|---|---|---|
| `youtube_videos.csv` | Un video de YouTube recolectado en el muestreo | `video_id` (293 valores únicos, sin faltantes) | `channel_id` apunta al canal propietario |
| `youtube_comments.csv` | Un comentario **principal** (no respuesta) publicado en un video | `comment_id` (406 valores únicos, sin faltantes) | `video_id` → videos; `author_channel_id` → autor |

**Variables relevantes por bloque de análisis:**

| Bloque | Variables de `videos` | Variables de `comments` |
|---|---|---|
| Identificación / red | `video_id`, `channel_id` | `comment_id`, `video_id`, `author_channel_id` |
| Etiquetas visibles | `title`, `channel_name`, `channel_handle` | `author_name`, `author_handle`, `video_title` |
| Cuantitativas | `view_count` | `like_count_text` → numérico, `reply_count` |
| Contenido / temas | `title`, `description`, `keywords` | `text` |
| Contexto de muestreo | `source_query`, `source_group`, `query_hits`, `dataset_sources`, `category` | `source_query`, `source_group`, `dataset_sources` |
| Temporal | `publish_date`, `upload_date`, `published_time` | `published_text` (relativo) |

### 1.3 Relación entre canal, video, autor, comentario, categoría y consulta

El esquema relacional observable es el siguiente:

```
 source_query / source_group  ──(procedimiento de muestreo)──►  video
           canal (channel_id) ──1:N──► video (video_id) ──1:N──► comentario (comment_id)
                                           │                          │
                                    category (1:1)          autor (author_channel_id)
```

- Un **canal** publica uno o más videos: `channel_id` → `video_id` es 1:N.
- Un **video** pertenece a exactamente una **categoría** de YouTube y recibe 0..N comentarios.
- Un **comentario** tiene exactamente un **autor** (`author_channel_id`); un autor puede escribir
  varios comentarios, en uno o varios videos, de uno o varios canales.
- La **consulta de búsqueda** (`source_query`, `source_group`) no es un atributo del contenido sino
  del *procedimiento de recolección*: describe cómo se encontró el video, no de qué trata.
- `channel_id` en el archivo de comentarios es el canal **dueño del video**, no el del autor: son
  espacios de identificadores distintos y no deben cruzarse.

La relación autor↔autor **no es observable**: sólo se infiere co-participación en un mismo video.

In [5]:
relaciones = pd.DataFrame([
    ["canal", "video", "1:N", "channel_id → video_id", f"{videos_raw['channel_id'].nunique()} canales publican {len(videos_raw)} videos"],
    ["video", "comentario", "1:N", "video_id → comment_id", f"{comentarios_raw['video_id'].nunique()} videos concentran {len(comentarios_raw)} comentarios"],
    ["autor", "comentario", "1:N", "author_channel_id → comment_id", f"{comentarios_raw['author_channel_id'].nunique()} autores escriben {len(comentarios_raw)} comentarios"],
    ["video", "categoría", "N:1", "video_id → category", f"{videos_raw['category'].nunique()} categorías distintas"],
    ["consulta", "video", "N:M", "source_query / query_hits", f"{videos_raw['source_query'].nunique()} consultas; un video puede aparecer en varias"],
    ["autor", "autor", "no observable", "—", "reply_count no identifica a quién respondió cada usuario"],
], columns=["origen", "destino", "cardinalidad", "llave", "evidencia"])
guardar_tabla(relaciones, "02_relaciones_entre_entidades")
relaciones

,origen,destino,cardinalidad,llave,evidencia
0,canal,video,1:N,channel_id → video_id,97 canales publican 293 videos
1,video,comentario,1:N,video_id → comment_id,19 videos concentran 406 comentarios
2,autor,comentario,1:N,author_channel_id → comment_id,332 autores escriben 406 comentarios
3,video,categoría,N:1,video_id → category,11 categorías distintas
4,consulta,video,N:M,source_query / query_hits,21 consultas; un video puede aparecer en varias
5,autor,autor,no observable,—,reply_count no identifica a quién respondió cada usuario


### 1.4 Integración por `video_id` y cobertura de la unión

In [6]:
comentarios_con_video = comentarios_raw["video_id"].isin(set(videos_raw["video_id"]))
integrado_raw = comentarios_raw.merge(
    videos_raw[["video_id", "title", "channel_id", "channel_name", "category",
                "source_group", "source_query", "view_count"]],
    on="video_id", how="left", suffixes=("", "_video"), validate="many_to_one",
)

videos_con_comentarios = int(comentarios_raw["video_id"].nunique())
videos_sin_comentarios = int(len(videos_raw) - videos_con_comentarios)

integracion = pd.DataFrame([
    ["Comentarios en el archivo", len(comentarios_raw)],
    ["Comentarios que sí se asociaron a un video", int(comentarios_con_video.sum())],
    ["Comentarios huérfanos (video_id sin catálogo)", int((~comentarios_con_video).sum())],
    ["Filas resultantes de la unión", len(integrado_raw)],
    ["Videos del catálogo", len(videos_raw)],
    ["Videos con al menos un comentario", videos_con_comentarios],
    ["Videos sin comentarios recolectados", videos_sin_comentarios],
    ["Cobertura de videos (%)", round(100 * videos_con_comentarios / len(videos_raw), 2)],
], columns=["indicador", "valor"])
guardar_tabla(integracion, "03_integracion_y_cobertura")

registrar("videos_con_comentarios", videos_con_comentarios)
registrar("videos_sin_comentarios", videos_sin_comentarios)
registrar("cobertura_videos_pct", round(100 * videos_con_comentarios / len(videos_raw), 2))
registrar("comentarios_asociados", int(comentarios_con_video.sum()))
registrar("comentarios_huerfanos", int((~comentarios_con_video).sum()))
registrar("canales_videos", int(videos_raw["channel_id"].nunique()))
registrar("canales_con_comentarios", int(comentarios_raw["channel_id"].nunique()))
registrar("autores_unicos", int(comentarios_raw["author_channel_id"].nunique()))
integracion

,indicador,valor
0,Comentarios en el archivo,406.00
1,Comentarios que sí se asociaron a un video,406.00
2,Comentarios huérfanos (video_id sin catálogo),0.00
3,Filas resultantes de la unión,406.00
4,Videos del catálogo,293.00
5,Videos con al menos un comentario,19.00
6,Videos sin comentarios recolectados,274.00
7,Cobertura de videos (%),6.48


## 2. Calidad, limpieza y preprocesamiento

### 2.1 Diagnóstico inicial de calidad

El diagnóstico cubre las seis dimensiones pedidas: dimensiones del archivo, tipo de cada variable,
valores faltantes, duplicados, variables constantes y valores atípicos, más un bloque específico de
consistencia entre identificadores, nombres y *handles*.

In [7]:
def contar_atipicos_iqr(serie: pd.Series):
    """Atípicos por la regla 1.5·IQR. Devuelve NA si la variable no es numérica."""
    if not pd.api.types.is_numeric_dtype(serie.dtype) or pd.api.types.is_bool_dtype(serie.dtype):
        return pd.NA
    valores = pd.to_numeric(serie, errors="coerce").dropna()
    if valores.empty:
        return 0
    q1, q3 = valores.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return 0
    return int(((valores < q1 - 1.5 * iqr) | (valores > q3 + 1.5 * iqr)).sum())


def diagnostico_calidad(frame: pd.DataFrame, archivo: str) -> pd.DataFrame:
    filas = []
    for columna in frame.columns:
        serie = frame[columna]
        no_nulos = serie.dropna()
        vacios_texto = 0
        if serie.dtype == object or str(serie.dtype) in {"string", "str"}:
            vacios_texto = int(serie.astype("string").fillna("").str.strip().eq("").sum())
        filas.append({
            "archivo": archivo,
            "variable": columna,
            "tipo_pandas": str(serie.dtype),
            "faltantes": int(serie.isna().sum()),
            "faltantes_pct": round(100 * serie.isna().mean(), 2),
            "vacios_o_espacios": vacios_texto,
            "valores_unicos": int(serie.nunique(dropna=True)),
            "es_constante": bool(len(no_nulos) > 0 and serie.nunique(dropna=True) <= 1),
            "es_vacia": bool(serie.isna().all()),
            "atipicos_iqr": contar_atipicos_iqr(serie),
            "ejemplo": acortar(no_nulos.iloc[0], 40) if len(no_nulos) else "",
        })
    return pd.DataFrame(filas)


calidad = pd.concat([
    diagnostico_calidad(videos_raw, "youtube_videos.csv"),
    diagnostico_calidad(comentarios_raw, "youtube_comments.csv"),
], ignore_index=True)
guardar_tabla(calidad, "04_diagnostico_calidad")
registrar("diagnostico_calidad", calidad.to_dict("records"))
calidad

,archivo,variable,tipo_pandas,faltantes,faltantes_pct,vacios_o_espacios,valores_unicos,es_constante,es_vacia,atipicos_iqr,ejemplo
0,youtube_videos.csv,video_id,string,0,0.00,0,293,False,False,<NA>,-5puKGEqcUc
1,youtube_videos.csv,title,str,0,0.00,0,274,False,False,<NA>,INSIVUMEH pronostica incremento de lluv…
2,youtube_videos.csv,channel_name,str,0,0.00,0,97,False,False,<NA>,T13 Noticias Guatemala
3,youtube_videos.csv,channel_id,string,0,0.00,0,97,False,False,<NA>,UCq0Cm-3SKthEySQc2JZBi1A
4,youtube_videos.csv,source_query,str,0,0.00,0,21,False,False,<NA>,guatemala lluvias
5,youtube_videos.csv,source_group,str,0,0.00,0,3,False,False,<NA>,topic
6,youtube_videos.csv,dataset_sources,str,0,0.00,0,23,False,False,<NA>,youtube_guatemala.csv | youtube_guatema…
7,youtube_videos.csv,channel_handle,string,0,0.00,0,97,False,False,<NA>,/@T13NoticiasGuatemala
8,youtube_videos.csv,published_time,str,13,4.44,13,80,False,False,<NA>,hace 2 días
9,youtube_videos.csv,view_count_text,str,13,4.44,13,259,False,False,<NA>,"2,390 vistas"


In [8]:
# Duplicados: exactos (fila completa) y por llave primaria.
duplicados = pd.DataFrame([
    ["youtube_videos.csv", "Filas exactamente duplicadas", int(videos_raw.duplicated().sum())],
    ["youtube_videos.csv", "video_id duplicado", int(videos_raw["video_id"].duplicated().sum())],
    ["youtube_videos.csv", "title duplicado (distinto video)", int(videos_raw["title"].duplicated().sum())],
    ["youtube_comments.csv", "Filas exactamente duplicadas", int(comentarios_raw.duplicated().sum())],
    ["youtube_comments.csv", "comment_id duplicado", int(comentarios_raw["comment_id"].duplicated().sum())],
    ["youtube_comments.csv", "texto duplicado (comment_id distinto)", int(comentarios_raw["text"].duplicated().sum())],
    ["youtube_comments.csv", "(video_id, author_channel_id, text) duplicado",
     int(comentarios_raw.duplicated(subset=["video_id", "author_channel_id", "text"]).sum())],
], columns=["archivo", "chequeo", "conteo"])
guardar_tabla(duplicados, "05_duplicados")
registrar("duplicados", duplicados.to_dict("records"))

constantes = calidad.query("es_constante or es_vacia")[["archivo", "variable", "valores_unicos", "faltantes", "es_constante", "es_vacia"]]
registrar("variables_constantes", constantes.to_dict("records"))
print(duplicados.to_string(index=False))
print("\nVariables constantes o completamente vacías:")
print(constantes.to_string(index=False))

             archivo                                       chequeo  conteo
  youtube_videos.csv                  Filas exactamente duplicadas       0
  youtube_videos.csv                            video_id duplicado       0
  youtube_videos.csv              title duplicado (distinto video)      19
youtube_comments.csv                  Filas exactamente duplicadas       0
youtube_comments.csv                          comment_id duplicado       0
youtube_comments.csv         texto duplicado (comment_id distinto)       2
youtube_comments.csv (video_id, author_channel_id, text) duplicado       2

Variables constantes o completamente vacías:
             archivo      variable  valores_unicos  faltantes  es_constante  es_vacia
youtube_comments.csv     is_pinned               1          0          True     False
youtube_comments.csv viewer_rating               0        406         False      True


In [9]:
# Consistencia entre identificadores, nombres visibles y handles.
def ids_con_varias_etiquetas(frame, col_id, col_etiqueta) -> int:
    conteo = frame.dropna(subset=[col_id]).groupby(col_id)[col_etiqueta].nunique(dropna=True)
    return int((conteo > 1).sum())


def etiquetas_con_varios_ids(frame, col_id, col_etiqueta) -> int:
    conteo = frame.dropna(subset=[col_etiqueta]).groupby(col_etiqueta)[col_id].nunique(dropna=True)
    return int((conteo > 1).sum())


consistencia = pd.DataFrame([
    ["videos", "channel_id → un solo channel_name", ids_con_varias_etiquetas(videos_raw, "channel_id", "channel_name")],
    ["videos", "channel_name → un solo channel_id", etiquetas_con_varios_ids(videos_raw, "channel_id", "channel_name")],
    ["videos", "channel_id → un solo channel_handle", ids_con_varias_etiquetas(videos_raw, "channel_id", "channel_handle")],
    ["videos", "channel_handle ≠ owner_handle", int((videos_raw["channel_handle"] != videos_raw["owner_handle"]).sum())],
    ["videos", "publish_date ≠ upload_date", int((videos_raw["publish_date"] != videos_raw["upload_date"]).sum())],
    ["videos", "video_url inconsistente con video_id",
     int((~videos_raw.apply(lambda r: str(r["video_id"]) in str(r["video_url"]), axis=1)).sum())],
    ["comentarios", "author_channel_id → un solo author_name", ids_con_varias_etiquetas(comentarios_raw, "author_channel_id", "author_name")],
    ["comentarios", "author_name → un solo author_channel_id", etiquetas_con_varios_ids(comentarios_raw, "author_channel_id", "author_name")],
    ["comentarios", "author_channel_id → un solo author_handle", ids_con_varias_etiquetas(comentarios_raw, "author_channel_id", "author_handle")],
    ["cruce", "channel_id de comentarios presente en videos",
     int(comentarios_raw["channel_id"].isin(set(videos_raw["channel_id"])).sum())],
    ["cruce", "author_channel_id que también es channel_id de un video",
     int(comentarios_raw["author_channel_id"].isin(set(videos_raw["channel_id"])).sum())],
    ["cruce", "video_title de comentarios ≠ title del catálogo",
     int((integrado_raw["video_title"] != integrado_raw["title"]).sum())],
    ["cruce", "source_group del comentario ≠ source_group del video",
     int((integrado_raw["source_group"] != integrado_raw["source_group_video"]).sum())],
    ["cruce", "source_query del comentario ≠ source_query del video",
     int((integrado_raw["source_query"] != integrado_raw["source_query_video"]).sum())],
], columns=["ámbito", "regla", "incumplimientos_o_conteo"])
guardar_tabla(consistencia, "06_consistencia_identificadores")
registrar("consistencia", consistencia.to_dict("records"))
consistencia

,ámbito,regla,incumplimientos_o_conteo
0,videos,channel_id → un solo channel_name,0
1,videos,channel_name → un solo channel_id,0
2,videos,channel_id → un solo channel_handle,0
3,videos,channel_handle ≠ owner_handle,0
4,videos,publish_date ≠ upload_date,0
5,videos,video_url inconsistente con video_id,0
6,comentarios,author_channel_id → un solo author_name,0
7,comentarios,author_name → un solo author_channel_id,0
8,comentarios,author_channel_id → un solo author_handle,0
9,cruce,channel_id de comentarios presente en videos,406


### 2.2 Variables que no pueden utilizarse o que requieren precaución

El diagnóstico anterior permite clasificar cada variable problemática y justificar su tratamiento.

In [10]:
likes_en_blanco = int(comentarios_raw["like_count_text"].fillna("").astype(str).str.strip().eq("").sum())
registrar("likes_en_blanco", likes_en_blanco)

problematicas = pd.DataFrame([
    ["viewer_rating", "Inutilizable", "406/406 faltantes (100 %); varianza nula.",
     "Se excluye de todo análisis. Se conserva en el archivo crudo para auditoría."],
    ["is_pinned", "Sin aporte", "Constante en False para los 406 registros.",
     "No se usa como variable explicativa; se documenta la ausencia de comentarios fijados."],
    ["published_time / published_text", "Precaución alta", "Tiempo relativo ('hace 2 días') dependiente del momento de recolección.",
     "No se convierte a fecha absoluta. Sólo se usa de forma ordinal y descriptiva."],
    ["view_count_text", "Redundante", "Texto con separador de miles y la palabra 'vistas'; 13 faltantes.",
     "Se usa view_count (entero) para todo cálculo; el texto queda como respaldo."],
    ["like_count_text", "Precaución", f"Almacenado como texto; {likes_en_blanco} registros son espacio en blanco.",
     "Se convierte a entero; el blanco se interpreta como 0 likes mostrados y se marca en like_count_imputado."],
    ["reply_count", "Precaución crítica", "Cuenta respuestas pero no identifica a sus autores.",
     "Nunca genera aristas entre usuarios. Se usa sólo como atributo de intensidad del comentario."],
    ["channel_name / author_name / handles", "No son identificadores", "Pueden repetirse o cambiar en el tiempo.",
     "Se conservan sólo como etiquetas; los IDs son la llave en toda la red."],
    ["source_query / source_group", "Sesgo de muestreo y homonimia", "Describen cómo se encontró el contenido, no su tema real. "
     "Además la variable existe en ambos archivos con el mismo nombre pero distinto significado y no coincide en 188 de 406 comentarios.",
     "Se usan sólo para describir el procedimiento. Al unir los archivos se conserva el sufijo _video "
     "para no confundir la ruta de recolección del video con la de sus comentarios."],
    ["description_snippet", "Redundante e incompleta", "Fragmento truncado de description; 25 faltantes.",
     "Se prefiere description para el análisis de contenido."],
    ["upload_date", "Redundante", "Idéntica a publish_date en el 100 % de los registros.",
     "Se conserva una sola variable temporal (publish_date)."],
    ["dataset_sources", "Procedencia", "Lista de archivos originales separados por '|'.",
     "Se usa para auditar la integración, no como variable de análisis."],
], columns=["variable", "clasificación", "problema_observado", "tratamiento_justificado"])
guardar_tabla(problematicas, "07_variables_problematicas")
registrar("variables_problematicas", problematicas.to_dict("records"))
problematicas

,variable,clasificación,problema_observado,tratamiento_justificado
0,viewer_rating,Inutilizable,406/406 faltantes (100 %); varianza nula.,Se excluye de todo análisis. Se conserva en el archivo crudo para auditoría.
1,is_pinned,Sin aporte,Constante en False para los 406 registros.,No se usa como variable explicativa; se documenta la ausencia de comentarios fijados.
2,published_time / published_text,Precaución alta,Tiempo relativo ('hace 2 días') dependiente del momento de recolección.,No se convierte a fecha absoluta. Sólo se usa de forma ordinal y descriptiva.
3,view_count_text,Redundante,Texto con separador de miles y la palabra 'vistas'; 13 faltantes.,Se usa view_count (entero) para todo cálculo; el texto queda como respaldo.
4,like_count_text,Precaución,Almacenado como texto; 189 registros son espacio en blanco.,Se convierte a entero; el blanco se interpreta como 0 likes mostrados y se marca en li...
5,reply_count,Precaución crítica,Cuenta respuestas pero no identifica a sus autores.,Nunca genera aristas entre usuarios. Se usa sólo como atributo de intensidad del comen...
6,channel_name / author_name / handles,No son identificadores,Pueden repetirse o cambiar en el tiempo.,Se conservan sólo como etiquetas; los IDs son la llave en toda la red.
7,source_query / source_group,Sesgo de muestreo y homonimia,"Describen cómo se encontró el contenido, no su tema real. Además la variable existe en...",Se usan sólo para describir el procedimiento. Al unir los archivos se conserva el sufi...
8,description_snippet,Redundante e incompleta,Fragmento truncado de description; 25 faltantes.,Se prefiere description para el análisis de contenido.
9,upload_date,Redundante,Idéntica a publish_date en el 100 % de los registros.,Se conserva una sola variable temporal (publish_date).


### 2.3 Normalización de identificadores y nombres

La normalización es **deliberadamente conservadora sobre los IDs**: sólo se recortan espacios
externos. No se cambian mayúsculas ni se sustituye ningún ID por un nombre visible, porque los IDs
de YouTube distinguen mayúsculas y minúsculas. Los nombres y *handles* sí se normalizan (Unicode
NFKC y espacios) porque son etiquetas de presentación.

In [11]:
def normalizar_id(serie: pd.Series) -> pd.Series:
    """Sólo recorta espacios externos: nunca altera el contenido del identificador."""
    return serie.astype("string").str.strip()


def normalizar_etiqueta(serie: pd.Series) -> pd.Series:
    """Normaliza Unicode y colapsa espacios en nombres visibles; preserva mayúsculas y acentos."""
    normalizada = serie.astype("string").map(
        lambda v: unicodedata.normalize("NFKC", v) if isinstance(v, str) else v
    )
    return normalizada.str.replace(r"\s+", " ", regex=True).str.strip()


RE_PORCENTAJE = re.compile(r"%[0-9A-Fa-f]{2}")


def normalizar_handle(serie: pd.Series) -> pd.Series:
    """Handles en forma canónica @nombre.

    Además del prefijo '/', se decodifica el porcentaje-encoding de URL: 14 handles de autor y 13 de
    canal llegan como '/@AlejandroP%C3%A9rez-b6r' en lugar de '@AlejandroPérez-b6r'. Sin decodificar,
    la misma persona podría mostrarse con dos etiquetas distintas en las tablas y figuras.
    """
    decodificado = serie.astype("string").map(
        lambda v: unquote(v) if isinstance(v, str) and RE_PORCENTAJE.search(v) else v)
    limpio = normalizar_etiqueta(decodificado).str.lstrip("/")
    return limpio.where(limpio.isna() | limpio.str.startswith("@"), "@" + limpio.fillna(""))


videos = videos_raw.copy()
comentarios = comentarios_raw.copy()

for columna in ["video_id", "channel_id"]:
    videos[columna] = normalizar_id(videos[columna])
for columna in ["video_id", "comment_id", "channel_id", "author_channel_id"]:
    comentarios[columna] = normalizar_id(comentarios[columna])

for columna in ["title", "channel_name", "category", "source_query", "source_group"]:
    videos[columna] = normalizar_etiqueta(videos[columna])
for columna in ["video_title", "channel_name", "author_name", "source_query", "source_group"]:
    comentarios[columna] = normalizar_etiqueta(comentarios[columna])

videos["channel_handle"] = normalizar_handle(videos["channel_handle"])
videos["owner_handle"] = normalizar_handle(videos["owner_handle"])
comentarios["author_handle"] = normalizar_handle(comentarios["author_handle"])

# Verificación: la normalización no debe fusionar ni romper identificadores.
handles_codificados = int(
    comentarios_raw["author_handle"].astype(str).str.contains(RE_PORCENTAJE, regex=True).sum()
    + videos_raw["channel_handle"].astype(str).str.contains(RE_PORCENTAJE, regex=True).sum())
registrar("handles_codificados", handles_codificados)

verificacion_ids = pd.DataFrame([
    ["video_id únicos antes/después", videos_raw["video_id"].nunique(), videos["video_id"].nunique()],
    ["channel_id únicos antes/después", videos_raw["channel_id"].nunique(), videos["channel_id"].nunique()],
    ["comment_id únicos antes/después", comentarios_raw["comment_id"].nunique(), comentarios["comment_id"].nunique()],
    ["author_channel_id únicos antes/después", comentarios_raw["author_channel_id"].nunique(), comentarios["author_channel_id"].nunique()],
    ["Handles con porcentaje-encoding de URL", handles_codificados, 0],
], columns=["chequeo", "antes", "después"])
guardar_tabla(verificacion_ids, "08_verificacion_normalizacion_ids")
registrar("verificacion_ids", verificacion_ids.to_dict("records"))
verificacion_ids

,chequeo,antes,después
0,video_id únicos antes/después,293,293
1,channel_id únicos antes/después,97,97
2,comment_id únicos antes/después,406,406
3,author_channel_id únicos antes/después,332,332
4,Handles con porcentaje-encoding de URL,27,0


### 2.4 Conversión a numérico de las variables de conteo almacenadas como texto

Decisiones documentadas:

- **Separadores de miles**: se eliminan `,` (formato en inglés) y `.` cuando actúa como separador
  de millares; también se eliminan espacios finos y no separables.
- **Abreviaturas**: se soportan los sufijos `K`, `M`, `B`, `MIL`, `MILL` y `M.` que YouTube muestra
  en algunas interfaces (por ejemplo `1.2 K` → 1 200).
- **Palabras de contexto**: `vistas`, `views`, `visualizaciones`, `reproducciones` se descartan.
- **Valores no válidos**: cadenas vacías, espacios y texto sin dígitos → `NA`, no 0.
- **Regla de negocio**: en `like_count_text` el blanco **sí** significa cero, porque YouTube oculta
  el contador cuando vale 0. Esa imputación queda registrada en `like_count_imputado`.

In [12]:
SUFIJOS = {"K": 1_000, "MIL": 1_000, "M": 1_000_000, "MILL": 1_000_000, "MM": 1_000_000, "B": 1_000_000_000}
RUIDO = re.compile(r"(vistas|views|visualizaciones|reproducciones|likes?|me gusta)", re.IGNORECASE)


def texto_a_numero(valor):
    """Convierte un conteo mostrado como texto a entero. Devuelve NA si no es interpretable."""
    if pd.isna(valor):
        return pd.NA
    texto = unicodedata.normalize("NFKC", str(valor))
    texto = RUIDO.sub("", texto)
    texto = texto.replace(" ", "").replace(" ", "").strip()
    if texto == "":
        return pd.NA
    coincidencia = re.match(r"^([\d.,\s]+)\s*([A-Za-z.]*)$", texto)
    if not coincidencia:
        return pd.NA
    numero, sufijo = coincidencia.groups()
    numero = numero.replace(" ", "")
    sufijo = sufijo.replace(".", "").upper()
    if "," in numero and "." in numero:           # 1.234,5 vs 1,234.5
        numero = numero.replace(".", "").replace(",", ".") if numero.rfind(",") > numero.rfind(".") \
            else numero.replace(",", "")
    elif "," in numero:
        partes = numero.split(",")
        numero = numero.replace(",", "") if all(len(p) == 3 for p in partes[1:]) else numero.replace(",", ".")
    elif "." in numero:
        partes = numero.split(".")
        if all(len(p) == 3 for p in partes[1:]) and len(partes) > 1 and not sufijo:
            numero = numero.replace(".", "")
    if numero in {"", ".", ","}:
        return pd.NA
    try:
        base = float(numero)
    except ValueError:
        return pd.NA
    return int(round(base * SUFIJOS.get(sufijo, 1)))


pruebas_conversion = pd.DataFrame(
    [(v, texto_a_numero(v)) for v in ["2,390 vistas", "1.234.567", "1.2 K", "3 M", " ", "", "45", "12 mil", "sin datos", None]],
    columns=["entrada", "salida"],
)
guardar_tabla(pruebas_conversion.astype(str), "09_pruebas_conversion_conteos")
print(pruebas_conversion.to_string(index=False))

     entrada  salida
2,390 vistas    2390
   1.234.567 1234567
       1.2 K    1200
         3 M 3000000
                <NA>
                <NA>
          45      45
      12 mil   12000
   sin datos    <NA>
         NaN    <NA>


In [13]:
comentarios["like_count_bruto"] = comentarios["like_count_text"].map(texto_a_numero).astype("Int64")
comentarios["like_count_imputado"] = comentarios["like_count_bruto"].isna()
comentarios["like_count"] = comentarios["like_count_bruto"].fillna(0).astype("int64")
comentarios["reply_count"] = pd.to_numeric(comentarios["reply_count"], errors="coerce").fillna(0).astype("int64")

videos["view_count"] = pd.to_numeric(videos["view_count"], errors="coerce").astype("Int64")
videos["view_count_text_num"] = videos["view_count_text"].map(texto_a_numero).astype("Int64")
coinciden = int((videos["view_count_text_num"].dropna() == videos.loc[videos["view_count_text_num"].notna(), "view_count"]).sum())
disponibles = int(videos["view_count_text_num"].notna().sum())

videos["publish_datetime"] = pd.to_datetime(videos["publish_date"], errors="coerce", utc=True)

# Magnitud de la discrepancia entre el texto mostrado y el conteo entero.
_dif = (videos["view_count_text_num"] - videos["view_count"]).dropna()
_dif_no_cero = _dif[_dif != 0].abs()
dif_mediana = int(_dif_no_cero.median()) if len(_dif_no_cero) else 0
dif_max = int(_dif_no_cero.max()) if len(_dif_no_cero) else 0
registrar("view_dif_mediana", dif_mediana)
registrar("view_dif_max", dif_max)

conversion = pd.DataFrame([
    ["like_count_text → like_count", f"{likes_en_blanco} blancos imputados a 0", int(comentarios['like_count'].sum())],
    ["view_count_text → verificación", f"{coinciden}/{disponibles} coinciden exactamente con view_count; "
     f"en las {disponibles - coinciden} discrepancias la diferencia mediana es de {dif_mediana} vistas (máx. {dif_max})",
     int(videos['view_count'].sum())],
    ["reply_count", "ya numérica; sin nulos", int(comentarios['reply_count'].sum())],
], columns=["conversión", "detalle", "total"])
guardar_tabla(conversion, "10_conversion_conteos")
registrar("view_text_coinciden", coinciden)
registrar("view_text_disponibles", disponibles)
registrar("total_likes", int(comentarios["like_count"].sum()))
registrar("total_respuestas", int(comentarios["reply_count"].sum()))
registrar("total_vistas", int(videos["view_count"].sum()))
conversion

,conversión,detalle,total
0,like_count_text → like_count,189 blancos imputados a 0,2325
1,view_count_text → verificación,227/280 coinciden exactamente con view_count; en las 53 discrepancias la diferencia me...,17706015
2,reply_count,ya numérica; sin nulos,51


### 2.5 – 2.6 `texto_original` y `texto_limpio`

Se conservan **dos versiones** del texto:

- `texto_original`: copia literal de `text`. Es la que se audita y la que alimenta el análisis de
  sentimiento, porque el modelo en español fue entrenado con texto natural (con acentos, signos y
  emojis) y destruirlos degradaría la predicción.
- `texto_limpio`: versión normalizada y lematizada, usada para frecuencias, bigramas y temas.

**Decisiones del pipeline de `texto_limpio`, en orden:**

| Paso | Decisión | Justificación |
|---|---|---|
| Unicode | NFKC + colapso de espacios | Unifica caracteres visualmente idénticos. |
| URLs | Se extraen y se eliminan | Aportan ruido léxico; se guardan en `urls_lista` para auditar. |
| Hashtags | Se separan a `hashtags_lista` y el término queda sin `#` | Permite contarlos aparte sin perder la palabra. |
| Menciones | Se separan a `menciones_lista` y se eliminan del texto | Un `@usuario` es un identificador, no vocabulario. |
| Emojis | Se extraen a `emojis_lista` y se eliminan del texto limpio | Se analizan por separado; sobreviven en `texto_original` para el sentimiento. |
| Minúsculas | Sí | Evita duplicar tipos por capitalización. |
| Números | Se eliminan tokens puramente numéricos | No aportan tema; las cifras relevantes viven en las variables de conteo. |
| Puntuación | Se elimina | Reduce ruido en frecuencias. |
| Stopwords | Lista de spaCy `es_core_news_sm` + lista propia de muletillas | Palabras funcionales del español no discriminan temas. |
| Lematización | spaCy `es_core_news_sm` | Une flexiones («corrupto/corruptos», «robar/roban»). |
| Tokens cortos | Se descartan los de 1–2 caracteres | Residuos de la limpieza. |

In [14]:
try:
    import spacy
    try:
        nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
    except OSError:
        from spacy.cli import download as _spacy_download
        _spacy_download("es_core_news_sm")
        nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
    SPACY_OK = True
except Exception as exc:                                       # pragma: no cover
    print(f"spaCy no disponible ({exc}); se usará lematización simplificada.")
    nlp, SPACY_OK = None, False

RE_URL = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
RE_HASHTAG = re.compile(r"#(\w+)", re.UNICODE)
RE_MENCION = re.compile(r"@([\w.\-]+)", re.UNICODE)
RE_EMOJI = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF" "\U0001F1E6-\U0001F1FF"
    "\U00002190-\U000021FF" "\U00002B00-\U00002BFF" "\U0000FE00-\U0000FE0F" "\U0001F900-\U0001F9FF" "]",
    flags=re.UNICODE,
)
RE_NO_PALABRA = re.compile(r"[^\wáéíóúüñÁÉÍÓÚÜÑ\s]", re.UNICODE)
RE_NUMERO = re.compile(r"\b\d+\b")

STOPWORDS_EXTRA = {
    "si", "ver", "va", "vas", "van", "ir", "solo", "solamente", "así", "aca", "acá", "allá", "ahi", "ahí",
    "q", "xq", "pq", "jaja", "jajaja", "jajajaja", "jeje", "d", "x", "k", "tan", "the", "and", "of",
    "ser", "estar", "haber", "hacer", "tener", "poder", "decir", "dar", "saber", "querer", "ah", "oh",
    "eh", "uy", "pue", "pues", "bien", "año", "años", "vez", "veces", "hoy", "ya", "aun", "aún",
    "él", "ella", "ellos", "ellas", "yo", "tú", "usted", "ustedes", "nosotros", "uno", "una",
    "este", "esta", "esto", "ese", "esa", "eso", "aquel", "cual", "cuales", "tal", "cosa",
}
STOPWORDS = set(nlp.Defaults.stop_words) | STOPWORDS_EXTRA if SPACY_OK else STOPWORDS_EXTRA


def separar_componentes(texto: str) -> dict:
    """Extrae URLs, hashtags, menciones y emojis antes de destruir el texto."""
    if not isinstance(texto, str):
        texto = ""
    return {
        "urls": RE_URL.findall(texto),
        "hashtags": [h.lower() for h in RE_HASHTAG.findall(texto)],
        "menciones": [m.lower() for m in RE_MENCION.findall(texto)],
        "emojis": RE_EMOJI.findall(texto),
    }


def prelimpiar(texto: str) -> str:
    """Normaliza y elimina URLs, menciones, emojis, números y puntuación. Conserva la palabra del hashtag."""
    if not isinstance(texto, str):
        return ""
    salida = unicodedata.normalize("NFKC", texto)
    salida = RE_URL.sub(" ", salida)
    salida = RE_MENCION.sub(" ", salida)
    salida = RE_HASHTAG.sub(r" \1 ", salida)       # se conserva la palabra sin el '#'
    salida = RE_EMOJI.sub(" ", salida)
    salida = salida.lower()
    salida = RE_NUMERO.sub(" ", salida)
    salida = RE_NO_PALABRA.sub(" ", salida)
    salida = re.sub(r"(\w)\1{2,}", r"\1\1", salida)  # 'holaaaa' → 'holaa'
    return re.sub(r"\s+", " ", salida).strip()


def sin_acentos(palabra: str) -> str:
    """Clave de agrupación que ignora tildes: 'país' y 'pais' comparten clave."""
    return "".join(c for c in unicodedata.normalize("NFD", palabra) if unicodedata.category(c) != "Mn")


STOPWORDS_SIN_ACENTOS = {sin_acentos(w) for w in STOPWORDS}


def token_util(pieza: str) -> bool:
    """Filtro final: descarta stopwords (con o sin tilde), números y tokens de 1–2 caracteres."""
    return (
        len(pieza) > 2
        and pieza not in STOPWORDS
        and sin_acentos(pieza) not in STOPWORDS_SIN_ACENTOS
        and not pieza.isdigit()
    )


def lematizar_lote(textos: list[str]) -> list[list[str]]:
    """Devuelve, por texto, la lista de lemas útiles.

    spaCy devuelve lemas multipalabra para los clíticos del español ('dárselo' → 'dar él').
    Cada lema se divide en sus piezas y el filtro de stopwords se aplica pieza por pieza;
    de lo contrario, pronombres como 'él' se colarían dentro de un lema compuesto.
    """
    if SPACY_OK:
        salida = []
        for doc in nlp.pipe(textos, batch_size=64):
            tokens = []
            for t in doc:
                if t.is_space or t.is_punct or t.like_num or t.text.lower() in STOPWORDS:
                    continue
                tokens.extend(p for p in t.lemma_.lower().split() if token_util(p))
            salida.append(tokens)
        return salida
    return [[t for t in texto.split() if token_util(t)] for texto in textos]


def construir_canonico(*corpus: list[list[str]]) -> dict:
    """Unifica variantes con y sin tilde ('pais' → 'país') eligiendo la forma más frecuente.

    Los usuarios escriben sin tildes de forma inconsistente; sin esta unificación el mismo
    concepto aparecería dos veces en las tablas de frecuencia y subestimaría su peso real.
    """
    conteo = Counter(t for lote in corpus for tokens in lote for t in tokens)
    grupos: dict[str, Counter] = {}
    for token, n in conteo.items():
        grupos.setdefault(sin_acentos(token), Counter())[token] = n
    return {t: max(g.items(), key=lambda kv: (kv[1], kv[0]))[0]
            for clave, g in grupos.items() for t in g}


def aplicar_canonico(lote: list[list[str]], mapa: dict) -> list[str]:
    return [" ".join(mapa.get(t, t) for t in tokens) for tokens in lote]


comentarios["texto_original"] = comentarios["text"].astype("string").fillna("")
_componentes = comentarios["texto_original"].map(separar_componentes)
comentarios["urls_lista"] = _componentes.map(lambda d: d["urls"])
comentarios["hashtags_lista"] = _componentes.map(lambda d: d["hashtags"])
comentarios["menciones_lista"] = _componentes.map(lambda d: d["menciones"])
comentarios["emojis_lista"] = _componentes.map(lambda d: d["emojis"])
comentarios["n_emojis"] = comentarios["emojis_lista"].map(len)
comentarios["texto_prelimpio"] = comentarios["texto_original"].map(prelimpiar)
_lemas_comentarios = lematizar_lote(comentarios["texto_prelimpio"].tolist())

# Mismo tratamiento para el contenido de los videos (título + descripción + keywords).
def parsear_lista_json(valor):
    if pd.isna(valor):
        return []
    try:
        cargado = json.loads(valor)
        return [str(x) for x in cargado] if isinstance(cargado, list) else [str(cargado)]
    except (json.JSONDecodeError, TypeError):
        return [p.strip() for p in str(valor).split("|") if p.strip()]


videos["keywords_lista"] = videos["keywords"].map(parsear_lista_json)
videos["query_hits_lista"] = videos["query_hits"].map(parsear_lista_json)
videos["n_keywords"] = videos["keywords_lista"].map(len)
videos["n_consultas"] = videos["query_hits_lista"].map(len)
videos["texto_original"] = (
    videos["title"].fillna("") + " " + videos["description"].fillna("")
).str.strip()
videos["hashtags_lista"] = videos["texto_original"].map(lambda t: separar_componentes(t)["hashtags"])
_lemas_videos = lematizar_lote(videos["texto_original"].map(prelimpiar).tolist())

# Unificación de variantes acentuadas, calculada sobre los dos corpus a la vez.
MAPA_CANONICO = construir_canonico(_lemas_comentarios, _lemas_videos)
comentarios["texto_limpio"] = aplicar_canonico(_lemas_comentarios, MAPA_CANONICO)
videos["texto_limpio"] = aplicar_canonico(_lemas_videos, MAPA_CANONICO)
comentarios["longitud_original"] = comentarios["texto_original"].str.len()
comentarios["tokens_limpios"] = comentarios["texto_limpio"].str.split().map(len)

print(comentarios[["texto_original", "texto_limpio"]].head(3).to_string())

                                                                                                                                                                    texto_original                                                                      texto_limpio
0                                                                                                         Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel                                          corrupto amigo viejo fiscal verbo cárcel
1  Están jóvenes porque no buscan un trabajo,  tuvieron suerte que no hay policías que le gusta del otro vando , por ir vestido de mujer se los hubieran ensamblado la maquinaria.  joven buscar trabajo suerte policía gustar var vestir mujer ensamblar maquinaria
2                                                                                Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, esto no es una maquila                            dejar gana 

### 2.7 Efecto cuantificado de la limpieza

In [15]:
vacios_antes = int(comentarios["texto_original"].str.strip().eq("").sum())
vacios_despues = int(comentarios["texto_limpio"].str.strip().eq("").sum())
modificados = int((comentarios["texto_original"].str.strip() != comentarios["texto_limpio"].str.strip()).sum())
dup_antes = int(comentarios.duplicated(subset=["texto_original"], keep=False).sum())
dup_despues = int(comentarios.duplicated(subset=["texto_limpio"], keep=False).sum())
tipos_antes = len({t for texto in comentarios["texto_prelimpio"] for t in texto.split()})
tipos_despues = len({t for texto in comentarios["texto_limpio"] for t in texto.split()})
tokens_antes = int(comentarios["texto_prelimpio"].str.split().map(len).sum())
tokens_despues = int(comentarios["tokens_limpios"].sum())

efecto = pd.DataFrame([
    ["Registros de entrada", len(comentarios_raw)],
    ["Registros conservados", len(comentarios)],
    ["Registros eliminados", len(comentarios_raw) - len(comentarios)],
    ["Textos modificados por la limpieza", modificados],
    ["Textos vacíos antes", vacios_antes],
    ["Textos vacíos después (quedan sin contenido léxico)", vacios_despues],
    ["Filas en grupos de texto duplicado antes", dup_antes],
    ["Filas en grupos de texto duplicado después", dup_despues],
    ["Tokens totales antes de stopwords/lematización", tokens_antes],
    ["Tokens totales después", tokens_despues],
    ["Vocabulario (tipos) antes", tipos_antes],
    ["Vocabulario (tipos) después", tipos_despues],
    ["Reducción de tokens (%)", round(100 * (1 - tokens_despues / max(tokens_antes, 1)), 1)],
    ["Comentarios con al menos un emoji", int((comentarios["n_emojis"] > 0).sum())],
    ["Comentarios con al menos una URL", int(comentarios["urls_lista"].map(len).gt(0).sum())],
    ["Comentarios con al menos una mención", int(comentarios["menciones_lista"].map(len).gt(0).sum())],
    ["Comentarios con al menos un hashtag", int(comentarios["hashtags_lista"].map(len).gt(0).sum())],
], columns=["métrica", "valor"])
guardar_tabla(efecto, "11_efecto_limpieza")
registrar("efecto_limpieza", efecto.to_dict("records"))
registrar("textos_vacios_despues", vacios_despues)
registrar("reduccion_tokens_pct", round(100 * (1 - tokens_despues / max(tokens_antes, 1)), 1))

# Ningún registro se elimina: perder un comentario destruiría una arista de la red.
comentarios["apto_para_texto"] = comentarios["texto_limpio"].str.strip().ne("")
efecto

,métrica,valor
0,Registros de entrada,406.0
1,Registros conservados,406.0
2,Registros eliminados,0.0
3,Textos modificados por la limpieza,405.0
4,Textos vacíos antes,0.0
5,Textos vacíos después (quedan sin contenido léxico),10.0
6,Filas en grupos de texto duplicado antes,4.0
7,Filas en grupos de texto duplicado después,19.0
8,Tokens totales antes de stopwords/lematización,9711.0
9,Tokens totales después,3769.0


### 2.8 Puntuación de sentimiento (insumo de la actividad 9)

El sentimiento se calcula aquí, junto con la limpieza, para que esté disponible en el análisis
exploratorio; su **interpretación** corresponde a la actividad 9.

**Herramienta elegida: `pysentimiento` (RoBERTuito), ajustado para español.** Justificación:

1. Es un modelo entrenado **en español**, no una traducción de un léxico en inglés como VADER o
   TextBlob. Con VADER, un comentario como *«qué gran robo»* quedaría sin puntuación porque sus
   palabras no están en el léxico inglés.
2. Fue entrenado con **texto de redes sociales** (corpus TASS de tuits), que comparte registro con
   los comentarios de YouTube: informal, corto, con errores ortográficos y emojis.
3. Es **contextual**: al basarse en un transformer capta negación e ironía mejor que un conteo de
   palabras positivas y negativas.

Se puntúa `texto_original`, no `texto_limpio`: la lematización elimina la negación («no»),
la puntuación y los emojis, que son precisamente las señales que el modelo necesita.

Si el modelo no está disponible se usa un **respaldo léxico en español con manejo de negación**,
para que el análisis siga siendo ejecutable; la variable `modelo_sentimiento` deja constancia
de cuál se utilizó.

In [16]:
LEXICO_POSITIVO = {
    "excelente", "bueno", "buena", "buenas", "buenos", "gracias", "felicidades", "felicitaciones",
    "genial", "increible", "increíble", "hermoso", "hermosa", "lindo", "linda", "amor", "amo",
    "mejor", "grande", "éxito", "exito", "exitos", "éxitos", "bendiciones", "orgullo", "orgulloso",
    "apoyo", "apoyamos", "feliz", "alegría", "alegria", "esperanza", "admirable", "maravilloso",
    "gran", "bien", "bonito", "bonita", "fantástico", "espectacular", "agradezco", "valiente",
}
LEXICO_NEGATIVO = {
    "corrupto", "corruptos", "corrupción", "corrupcion", "ladrón", "ladron", "ladrones", "robo",
    "roban", "robar", "malo", "mala", "malos", "pésimo", "pesimo", "horrible", "vergüenza",
    "verguenza", "asco", "basura", "mentira", "mentiroso", "estafa", "criminal", "delincuente",
    "odio", "triste", "peor", "fracaso", "impunidad", "narco", "traidor", "sinvergüenza",
    "sinverguenza", "incapaz", "inútil", "inutil", "cárcel", "carcel", "hambre", "pobreza",
    "burla", "engaño", "engano", "abuso", "injusticia", "miedo", "terror", "payaso",
}
NEGADORES = {"no", "ni", "nunca", "jamás", "jamas", "nada", "tampoco", "sin"}


def sentimiento_lexico(texto: str) -> tuple[str, float]:
    """Respaldo: puntaje = (pos - neg) / (pos + neg), invirtiendo la polaridad tras un negador."""
    tokens = re.findall(r"[\wáéíóúüñÁÉÍÓÚÜÑ]+", str(texto).lower())
    positivos = negativos = 0
    negado = False
    for token in tokens:
        polaridad = 1 if token in LEXICO_POSITIVO else (-1 if token in LEXICO_NEGATIVO else 0)
        if polaridad and negado:
            polaridad *= -1
        if polaridad > 0:
            positivos += 1
        elif polaridad < 0:
            negativos += 1
        negado = token in NEGADORES
    total = positivos + negativos
    if total == 0:
        return "NEU", 0.0
    puntaje = (positivos - negativos) / total
    etiqueta = "POS" if puntaje > 0.2 else ("NEG" if puntaje < -0.2 else "NEU")
    return etiqueta, float(puntaje)


MODELO_SENTIMIENTO = "respaldo léxico en español con negación"
try:
    from pysentimiento import create_analyzer
    analizador = create_analyzer(task="sentiment", lang="es")
    salidas = analizador.predict(comentarios["texto_original"].tolist())
    comentarios["sentimiento"] = [s.output for s in salidas]
    comentarios["prob_pos"] = [float(s.probas["POS"]) for s in salidas]
    comentarios["prob_neg"] = [float(s.probas["NEG"]) for s in salidas]
    comentarios["prob_neu"] = [float(s.probas["NEU"]) for s in salidas]
    comentarios["puntaje_sentimiento"] = comentarios["prob_pos"] - comentarios["prob_neg"]
    comentarios["confianza_sentimiento"] = [float(max(s.probas.values())) for s in salidas]
    MODELO_SENTIMIENTO = "pysentimiento / RoBERTuito (robertuito-sentiment-analysis, español)"
except Exception as exc:                                        # pragma: no cover
    print(f"pysentimiento no disponible ({type(exc).__name__}); se usa el respaldo léxico.")
    resultado = comentarios["texto_original"].map(sentimiento_lexico)
    comentarios["sentimiento"] = resultado.map(lambda r: r[0])
    comentarios["puntaje_sentimiento"] = resultado.map(lambda r: r[1])
    comentarios["confianza_sentimiento"] = comentarios["puntaje_sentimiento"].abs()
    for columna in ["prob_pos", "prob_neg", "prob_neu"]:
        comentarios[columna] = np.nan

registrar("modelo_sentimiento", MODELO_SENTIMIENTO)
distribucion_sentimiento = comentarios["sentimiento"].value_counts().reindex(["POS", "NEU", "NEG"]).fillna(0).astype(int)
registrar("sentimiento_global", distribucion_sentimiento.to_dict())
registrar("sentimiento_medio", round(float(comentarios["puntaje_sentimiento"].mean()), 3))
print(f"Modelo: {MODELO_SENTIMIENTO}")
print(distribucion_sentimiento.to_string())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6516.92it/s]

Map:   0%|          | 0/406 [00:00<?, ? examples/s]

Map: 100%|██████████| 406/406 [00:00<00:00, 12658.61 examples/s]

Modelo: pysentimiento / RoBERTuito (robertuito-sentiment-analysis, español)
sentimiento
POS     78
NEU     79
NEG    249


## Cierre del hito 1

La integración es íntegra y las llaves son utilizables: ningún comentario queda huérfano y la unión
no expande filas. El riesgo real está en la cobertura, no en la calidad de las llaves.

La limpieza conserva los dos textos exigidos —`texto_original` para auditoría y sentimiento,
`texto_limpio` para frecuencias y temas— y no elimina ningún registro, porque cada comentario es
una arista de la red que se construye en el hito 2.

El análisis exploratorio, la red bipartita y las actividades 5 a 10 continúan en
`02_exploratorio_y_red_bipartita.ipynb` y `03_laboratorio6_completo.ipynb`.